In [2]:
import gzip
import itertools
from collections import defaultdict 

## Load in the palindrome library, and make a mapping of target to sequence

In [3]:
palendromes = {}

with open("synth_constructs_2018_08_28.fasta") as f:
    for line1,line2 in itertools.zip_longest(*[f]*2):
        pal_id =  line1.split('_')[1][0:20]
        pal_seq = line2.upper()
        # if pal_id in palendromes:
        #    print("Warning: collision on " + pal_id)
        palendromes[pal_id] = pal_seq
print(len(palendromes))     

314


### Now process our sequencing data

first, a function that takes the reference and sequence alignments and extracts the guide/target pairs


In [4]:
### returns a tuple of the extracted guide and target
def process_ref_seq_alignment(ref,sequence,guide_ref_start=20,guide_ref_stop=40,target_ref_start=128,target_ref_stop=154):
    ref_pos = 0
    guide = ""
    target = ""
    for ref_base,seq_base in zip(ref,sequence):
        if ref_pos >= guide_ref_start and ref_pos < guide_ref_stop:
            guide += seq_base
        elif ref_pos >= target_ref_start and ref_pos < target_ref_stop:
            target += seq_base
        if ref_base != '-':
            ref_pos += 1
    return((guide,target))


##### now for each pair of sequences (over multiple lines) in the aligned file, count outcomes, removing any misaligned ( containing dashes) sequences

In [5]:
ref = ""
sequence = ""
ref_name_set = False
seq_name_set = False

palendromes_to_outcomes = defaultdict(list)

with gzip.open('aligned_sequences.fasta.gz', 'rt') as f:
    for line in f:
        if line.startswith('>'):
            if not ref_name_set:
                ref_name_set = True
            elif (not seq_name_set) and ref_name_set:
                seq_name_set = True
            elif seq_name_set and ref_name_set:
                (guide,target) = process_ref_seq_alignment(ref,sequence)
                #if not ('-' in ref or '-' in sequence):
                palendromes_to_outcomes[guide].append(target)
                ref = ""
                sequence = ""
                ref_name_set = False
                seq_name_set = False
            else:
                print("WE SHOULDNT BE HERE")
        else:
            if ref_name_set and (not seq_name_set):
                ref += line.strip("\n")
            else:
                sequence += line.strip("\n")

#### Now match up each read's target sequencing and look for base editing

In [6]:
def target_pileup(guide,targets,target_max_diff=0.40):
    alt_base = [0] * len(guide)
    ref_base = [0] * len(guide)
    
    for target in targets:
        target_sliced = target[3:23]
        differences = [0 if (x == y) else 1 for (x,y) in zip(guide,target_sliced)]
        prop_diff = float(sum(differences))/float(len(guide))
        
        if prop_diff < target_max_diff:
            for index,diff in enumerate(differences):
                if diff == 0:
                    ref_base[index] += 1
                elif diff == 1:
                    alt_base[index] += 1
                else:
                    print("It's also bad if we end up here")
    
    count = alt_base[0] + ref_base[0]
    editing = [float(a)/float(a+r+0.0000001) for (a,r) in zip(alt_base,ref_base)]
    return((count,editing))

target_pileup("GTATAGGTGATCACCTATAC",["CCGGTATAGGTGGTCACCTATACCGG"])

(1,
 [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.9999999000000099,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0])

#### now output the base-editing counts per position

In [7]:
pal_count = 0
matching_reads = 0

output = open("palindromes_base_counts.txt","w")
output.write("guide\ttarget_count\t" + "\t".join(["base" + str(x) for x in range(1,21)]) + "\n")

for key,vl in palendromes.items():
    if key in palendromes_to_outcomes:
        pal_count += 1
        matching_reads += len(palendromes_to_outcomes[key])
        count,editing = target_pileup(key,palendromes_to_outcomes[key])
        output.write(key + "\t" + str(count) + "\t" + "\t".join([str(x) for x in editing]) + "\n")
        
print(pal_count)
print(matching_reads)
output.close()

310
405313


In [16]:
# palendromes_to_outcomes["GGTGACCGGTACCGGTCACC"]